# 02 - Transformación y limpieza de datos
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

Este notebook toma el fichero crudo descargado en el paso anterior y produce un dataset limpio, tipado y listo para cargar en Snowflake.

**Entrada:** `data/raw/contratos_menores.xlsx`  
**Salida:** `data/processed/contratos_limpios.csv`

---

### Decisiones y supuestos documentados

| # | Decisión | Justificación |
|---|---|---|
| 1 | Se eliminan `Unnamed: 0` y `Lista de lotes` | 100% y 99.8% nulas respectivamente. Sin valor analítico. |
| 2 | `Código CPV` (96% nulo) se conserva tal cual | Podría ser útil para análisis futuros; se anota el alto % de nulos. |
| 3 | Se separa `Contratistas` en `nif_contratista` y `nombre_contratista` | El campo mezcla CIF/NIF con nombre en un solo string (`"B27188317 - EXCAVACIONES J.CARREIRA"`). |
| 4 | Los importes (strings con formato español `"1.234,56"`) se convierten a `float` | Requiere quitar los puntos de miles y sustituir la coma decimal por punto. |
| 5 | Los nombres de columna se normalizan a snake_case sin tildes ni espacios | Facilita el trabajo en SQL y Python. |
| 6 | `Fecha formalización` (96% nula) se sustituye por año extraído del número de referencia | El número de referencia sigue el patrón `CT NNN/AAAA`, por lo que el año es recuperable. |
| 7 | Se eliminan los registros `Contrato Mayor` | El proyecto es específicamente de contratos menores; los contratos mayores son ruido. |

## 0 — Importaciones y rutas

In [ ]:
import re          # expresiones regulares — para extraer el año del número de referencia
import pandas as pd
from pathlib import Path  # manejo de rutas independiente del sistema operativo

# Rutas de entrada y salida del proceso de transformación
RUTA_RAW       = Path("../data/raw/contratos_menores.xlsx")
RUTA_PROCESADO = Path("../data/processed/contratos_limpios.csv")

print(f"Leyendo: {RUTA_RAW}")

## 1 — Cargar el fichero crudo

In [ ]:
# dtype=str fuerza que todos los campos se lean como texto puro,
# evitando que pandas convierta CIFs como "07123456A" en NaN o los trunque
df = pd.read_excel(RUTA_RAW, dtype=str)

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
df.head(3)

## 2 — Eliminar columnas sin valor

In [ ]:
# Columnas con 100% y 99.8% de nulos — no aportan información útil al análisis
COLUMNAS_BASURA = ["Unnamed: 0", "Lista de lotes"]
df = df.drop(columns=COLUMNAS_BASURA)

print(f"Columnas tras limpieza: {df.shape[1]}")
print(df.columns.tolist())

## 3 — Renombrar columnas a snake_case

In [ ]:
# Mapeamos los nombres originales del Excel (con espacios, tildes y paréntesis)
# a nombres snake_case que funcionan directamente en SQL y como atributos de Python
RENOMBRAR = {
    "Entidad Contratante":                               "entidad_contratante",
    "Número de Referencia del Contrato":                  "num_referencia",
    "Objeto del Contrato":                               "objeto_contrato",
    "Valor estimado del contrato (en euros)":             "valor_estimado_eur",
    "Presupuesto base sin impuestos (en euros)":          "presupuesto_base_eur",
    "Plazo ejecución (en meses)":                        "plazo_meses",
    "Código CPV del objeto del contrato":                 "codigo_cpv",
    "Tipo de contratación":                              "tipo_contratacion",
    "Tipo de Contrato":                                  "tipo_contrato",
    "Tipo de procedimiento":                             "tipo_procedimiento",
    "Sistema de contratación":                           "sistema_contratacion",
    "Tramitación":                                       "tramitacion",
    "Fecha formalización":                               "fecha_formalizacion",
    "Importe total ofertado (sin impuestos) (en euros)": "importe_sin_iva_eur",
    "Importe total ofertado (con impuestos) (en euros)": "importe_con_iva_eur",
    "URL a la licitación específica del expediente":      "url_licitacion",
    "Observaciones":                                     "observaciones",
    "Contratistas":                                      "contratistas_raw",
}

df = df.rename(columns=RENOMBRAR)
print(df.columns.tolist())

## 4 — Limpiar e importar la columna de contratistas

Formato original: `"B27188317 - EXCAVACIONES J.CARREIRA, S.L."`  
Se separa por el primer ` - ` en NIF/CIF y nombre.

In [ ]:
def separar_contratista(texto):
    """Devuelve (nif, nombre) a partir de 'NIF - NOMBRE'."""
    if pd.isna(texto) or not isinstance(texto, str):
        return pd.NA, pd.NA
    # maxsplit=1 divide solo por el primer " - ", evitando romper nombres que contengan guion
    partes = texto.split(" - ", maxsplit=1)
    if len(partes) == 2:
        return partes[0].strip(), partes[1].strip()
    # Si no hay separador, devolvemos el texto completo como nombre sin NIF
    return pd.NA, texto.strip()

# Aplicamos la función fila a fila y expandimos el resultado en dos columnas nuevas
df[["nif_contratista", "nombre_contratista"]] = (
    df["contratistas_raw"]
    .apply(lambda x: pd.Series(separar_contratista(x)))
)

print("Ejemplos de separación:")
df[["contratistas_raw", "nif_contratista", "nombre_contratista"]].head(5)

## 5 — Convertir importes de string español a float

Formato español: `"1.234,56"` → `1234.56`  
Proceso: quitar puntos de miles, sustituir coma decimal por punto, convertir a float.

In [ ]:
def euros_a_float(valor):
    """Convierte un string de importe en formato español a float."""
    if pd.isna(valor) or str(valor).strip() in ("", "nan"):
        return pd.NA
    # Eliminamos el punto de miles y sustituimos la coma decimal por punto
    # Ejemplo: "1.234,56" → "1234.56"
    limpio = str(valor).strip().replace(".", "").replace(",", ".")
    try:
        return float(limpio)
    except ValueError:
        # Si el valor contiene texto inesperado devolvemos nulo en lugar de fallar
        return pd.NA

COLS_IMPORTE = ["valor_estimado_eur", "presupuesto_base_eur", "importe_sin_iva_eur", "importe_con_iva_eur"]

# Float64 con mayúscula es el tipo nullable de pandas — admite NaN, a diferencia de float64 de numpy
for col in COLS_IMPORTE:
    df[col] = df[col].apply(euros_a_float).astype("Float64")

print("Tipos resultantes:")
print(df[COLS_IMPORTE].dtypes)
print("\nEjemplos:")
df[COLS_IMPORTE].head()

## 6 — Extraer el año del número de referencia

`Fecha formalización` tiene un 96% de nulos, pero el **número de referencia** incluye siempre el año:  
`CT 80/2022-e` → **2022** | `CT 02/2023` → **2023**

Extraemos ese año como columna `anio_contrato`, que usaremos para análisis temporales.

In [ ]:
def extraer_anio(referencia):
    """Extrae el año de 4 dígitos de una referencia tipo 'CT 80/2022-e'."""
    if pd.isna(referencia):
        return pd.NA
    # Buscamos una barra "/" seguida de exactamente 4 dígitos
    match = re.search(r"/(\d{4})", str(referencia))
    # group(1) devuelve el contenido del primer paréntesis capturador del patrón
    return int(match.group(1)) if match else pd.NA

# Int64 con mayúscula permite nulos; int64 estándar no admite NaN
df["anio_contrato"] = df["num_referencia"].apply(extraer_anio).astype("Int64")

print("Distribución por año:")
print(df["anio_contrato"].value_counts().sort_index())
print(f"\nReferencias sin año extraíble: {df['anio_contrato'].isna().sum()}")

## 7 — Normalizar `fecha_formalizacion`

Solo un 4% de registros tiene fecha. La convertimos a `datetime` cuando existe.

In [ ]:
# dayfirst=True indica que el formato es DD-MM-AAAA (formato español)
# errors="coerce" convierte a NaT (nulo de fecha) los valores que no se puedan parsear
df["fecha_formalizacion"] = pd.to_datetime(df["fecha_formalizacion"], dayfirst=True, errors="coerce")

con_fecha = df["fecha_formalizacion"].notna().sum()
print(f"Registros con fecha_formalizacion: {con_fecha} ({con_fecha / len(df) * 100:.1f}%)")
df.loc[df["fecha_formalizacion"].notna(), ["num_referencia", "fecha_formalizacion"]].head()

## 8 — Filtrar: quedarse solo con contratos menores

In [ ]:
antes = len(df)
# Filtramos para quedarnos solo con los contratos menores
df = df[df["tipo_contratacion"].str.strip() == "Contrato Menor"].copy()

print(f"Registros eliminados (Contrato Mayor): {antes - len(df)}")
print(f"Registros restantes (Contrato Menor):  {len(df)}")

## 9 — Convertir `plazo_meses` a numérico

In [ ]:
# errors="coerce" convierte a NaN los valores que no sean numéricos en lugar de fallar
df["plazo_meses"] = pd.to_numeric(df["plazo_meses"], errors="coerce").astype("Float64")

print(df["plazo_meses"].describe())

## 10 — Marcar contratos cercanos al límite legal

Los contratos menores tienen límites legales:  
- Servicios y suministros: **≤ 15.000 €**  
- Obras: **≤ 40.000 €**

Añadimos `flag_limite` para detectar contratos que superan o rozan esos umbrales.

In [ ]:
LIMITE_SERVICIOS = 15_000   # límite legal para servicios y suministros (€)
LIMITE_OBRAS     = 40_000   # límite legal para obras (€)
UMBRAL_ALERTA    = 0.90     # marcamos alerta si el importe supera el 90% del límite

def flag_limite(row):
    importe = row["presupuesto_base_eur"]
    tipo    = str(row["tipo_contrato"]).strip()
    if pd.isna(importe):
        return "sin_importe"
    # Las obras tienen un límite distinto al resto de tipos de contrato
    limite = LIMITE_OBRAS if tipo == "Obras" else LIMITE_SERVICIOS
    if importe > limite:
        return "supera_limite"       # contrato que incumple la ley de contratos
    if importe >= limite * UMBRAL_ALERTA:
        return "cerca_del_limite"    # posible fraccionamiento artificial del contrato
    return "ok"

# axis=1 aplica la función fila a fila (no columna a columna)
df["flag_limite"] = df.apply(flag_limite, axis=1)

print("Distribución de flags:")
print(df["flag_limite"].value_counts())

print("\nContratos que superan el límite legal:")
df.loc[
    df["flag_limite"] == "supera_limite",
    ["num_referencia", "objeto_contrato", "tipo_contrato", "presupuesto_base_eur", "nombre_contratista"]
]

## 11 — Revisión final de calidad

In [ ]:
# Tabla resumen con tipo de dato, nulos y valores únicos de cada columna del dataset final
resumen = pd.DataFrame({
    "tipo":    df.dtypes,
    "nulos":   df.isnull().sum(),
    "% nulos": (df.isnull().mean() * 100).round(1),
    "únicos":  df.nunique(),  # valores distintos — útil para detectar columnas constantes
})
print(resumen.to_string())

In [ ]:
# Estadísticas descriptivas de las columnas de importe (mín, máx, media, percentiles...)
print("=== Estadísticas de importes ===")
df[COLS_IMPORTE].describe().round(2)

## 12 — Ordenar columnas y guardar

In [ ]:
# Reordenamos las columnas en grupos lógicos para facilitar la lectura en Snowflake y Power BI
ORDEN_COLUMNAS = [
    # Identificación
    "num_referencia",
    "anio_contrato",
    "fecha_formalizacion",
    "entidad_contratante",
    # Clasificación
    "tipo_contrato",
    "tipo_procedimiento",
    "sistema_contratacion",
    "tramitacion",
    # Descripción
    "objeto_contrato",
    "codigo_cpv",
    "plazo_meses",
    # Importes
    "valor_estimado_eur",
    "presupuesto_base_eur",
    "importe_sin_iva_eur",
    "importe_con_iva_eur",
    "flag_limite",
    # Contratista
    "nif_contratista",
    "nombre_contratista",
    "contratistas_raw",   # mantenemos el campo original como trazabilidad
    # Extra
    "observaciones",
    "url_licitacion",
]

df = df[ORDEN_COLUMNAS]

# encoding="utf-8-sig" añade BOM al inicio del fichero para que Excel y Power BI
# abran correctamente los caracteres especiales (tildes, ñ...)
df.to_csv(RUTA_PROCESADO, index=False, encoding="utf-8-sig")

print(f"Guardado en: {RUTA_PROCESADO}")
print(f"Filas: {len(df)}  |  Columnas: {len(df.columns)}")

## Resumen del paso de transformación

| Acción | Resultado |
|---|---|
| Columnas eliminadas | 2 (`Unnamed: 0`, `Lista de lotes`) |
| Columnas renombradas | 18 → snake_case |
| Columnas nuevas | 4 (`nif_contratista`, `nombre_contratista`, `anio_contrato`, `flag_limite`) |
| Importes convertidos | 4 columnas → `Float64` |
| Fecha parseada | `fecha_formalizacion` → `datetime64` |
| Contratos mayores eliminados | registros con `Tipo de contratación = Contrato Mayor` |
| Fichero de salida | `data/processed/contratos_limpios.csv` |